# 01 — Data Collection and Integration

**Course:** COMP4381 — Data Science and Analytics  
**Semester:** Spring 2026  
**Owner:** Abdullah Mattour (1223061)  

---

## Purpose

This notebook downloads all raw datasets, filters them to our five target countries,
merges them into a single analytical table, and engineers the features we need for
modelling. The output is a single CSV file (`data/processed/merged_raw.csv`) that
the cleaning notebook will pick up next.

**Data sources:**

| Source | What it provides | Download method |
|--------|-----------------|------------------|
| EUROCONTROL — Airport Arrival ATFM Delays | Daily ATFM delay minutes, flights delayed >15 min, delay by cause | Stable URL (`.csv.bz2`) |
| EUROCONTROL — Airport Traffic | Daily IFR departures and arrivals per airport | Stable URL (`.csv`) |
| OurAirports — airports.csv | Airport metadata: type, lat, lon, elevation, region | Stable URL |
| OurAirports — countries.csv | ISO country code to country name mapping | Stable URL |
| OurAirports — runways.csv | Runway length, surface type, count per airport | Stable URL |

## 1. Imports and Constants

In [1]:
import pandas as pd
import numpy as np
import requests
import os
from pathlib import Path

# --- Paths ---
RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# --- OurAirports URLs (stable GitHub Pages mirror) ---
AIRPORTS_URL = "https://davidmegginson.github.io/ourairports-data/airports.csv"
COUNTRIES_URL = "https://davidmegginson.github.io/ourairports-data/countries.csv"
RUNWAYS_URL = "https://davidmegginson.github.io/ourairports-data/runways.csv"

# --- EUROCONTROL URLs (direct CSV downloads from ANS Performance portal) ---
ATFM_DLY_URLS = [
    "https://www.eurocontrol.int/performance/data/download/csv/apt_dly_2023.csv.bz2",
    "https://www.eurocontrol.int/performance/data/download/csv/apt_dly_2024.csv.bz2",
]
TRAFFIC_URLS = [
    "https://www.eurocontrol.int/performance/data/download/csv/airport_traffic_2023.csv",
    "https://www.eurocontrol.int/performance/data/download/csv/airport_traffic_2024.csv",
]

# --- Target countries (EUROCONTROL uses full English names) ---
TARGET_COUNTRIES = ["United Kingdom", "Germany", "France", "Netherlands", "Spain"]

# --- ISO codes for OurAirports filtering ---
TARGET_ISO = ["GB", "DE", "FR", "NL", "ES"]

# Browser-style header so static hosts don't reject us
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/124.0.0.0 Safari/537.36"
    )
}

print("Setup complete.")

Setup complete.


## 2. Download OurAirports data

We download three CSV files from OurAirports. These are hosted on GitHub Pages
and are updated daily, so every run gets the latest airport metadata.

In [2]:
def download_csv(url, save_path, headers=HEADERS):
    """Download a CSV from a URL and save it locally."""
    print(f"Downloading {url.split('/')[-1]} ... ", end="")
    resp = requests.get(url, headers=headers, timeout=60)
    resp.raise_for_status()
    save_path.write_bytes(resp.content)
    size_kb = len(resp.content) / 1024
    print(f"OK ({size_kb:.0f} KB)")

download_csv(AIRPORTS_URL, RAW_DIR / "airports.csv")
download_csv(COUNTRIES_URL, RAW_DIR / "countries.csv")
download_csv(RUNWAYS_URL, RAW_DIR / "runways.csv")

OK (12343 KB)

OK (24 KB)

OK (3856 KB)


## 3. Load OurAirports into DataFrames

In [3]:
airports_raw = pd.read_csv(RAW_DIR / "airports.csv")
countries_raw = pd.read_csv(RAW_DIR / "countries.csv")
runways_raw = pd.read_csv(RAW_DIR / "runways.csv")

print(f"airports : {airports_raw.shape}")
print(f"countries: {countries_raw.shape}")
print(f"runways  : {runways_raw.shape}")
airports_raw.head(3)

airports : (85473, 19)
countries: (249, 6)
runways  : (47950, 20)


,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,icao_code,iata_code,gps_code,local_code,home_link,wikipedia_link,keywords
0,6523,00A,heliport,Total RF Heliport,40.070985,-74.933689,11.0,NaN,US,US-PA,Bensalem,no,NaN,NaN,K00A,00A,https://www.penndot.pa.gov/TravelInPA/airports...,NaN,NaN
1,323361,00AA,small_airport,Aero B Ranch Airport,38.704022,-101.473911,3435.0,NaN,US,US-KS,Leoti,no,NaN,NaN,00AA,00AA,NaN,NaN,NaN
2,6524,00AK,small_airport,Lowell Field,59.947733,-151.692524,450.0,NaN,US,US-AK,Anchor Point,no,NaN,NaN,00AK,00AK,NaN,NaN,NaN


## 4. Filter airports to our five target countries

We keep only large and medium airports, since small airstrips, heliports, and
seaplane bases do not appear in the EUROCONTROL punctuality data and would
just create unmatched rows after the merge.

In [4]:
# Filter by country and airport type
keep_types = ["large_airport", "medium_airport"]

airports = airports_raw[
    (airports_raw["iso_country"].isin(TARGET_ISO)) &
    (airports_raw["type"].isin(keep_types))
].copy()

print(f"Airports after filtering: {len(airports)}")
print()
print("Per country:")
print(airports.groupby("iso_country")["ident"].count())
print()

# Keep only the columns we need
airport_cols = [
    "ident", "type", "name", "latitude_deg", "longitude_deg",
    "elevation_ft", "iso_country", "iso_region", "municipality",
    "scheduled_service", "iata_code",
]
airports = airports[airport_cols].copy()

# Rename 'ident' to 'icao_code' for clarity
airports = airports.rename(columns={"ident": "icao_code"})

# Convert scheduled_service from yes/no to 1/0
airports["scheduled_service"] = (airports["scheduled_service"] == "yes").astype(int)

airports.head()

Airports after filtering: 368

Per country:
iso_country
DE     75
ES     55
FR    135
GB     89
NL     14
Name: ident, dtype: int64



,icao_code,type,name,latitude_deg,longitude_deg,elevation_ft,iso_country,iso_region,municipality,scheduled_service,iata_code
23326,EDAC,medium_airport,Leipzig–Altenburg Airport,50.981945,12.506389,640.0,DE,DE-TH,Nobitz,0,AOC
23330,EDAH,medium_airport,Heringsdorf Airport,53.878700,14.152300,94.0,DE,DE-MV,Zirchow,1,HDF
23359,EDBN,medium_airport,Neubrandenburg Trollenhagen Airport,53.602200,13.306000,228.0,DE,DE-MV,Trollenhagen,0,FNB
23363,EDBR,medium_airport,Rothenburg/Görlitz Airfield,51.363335,14.950000,515.0,DE,DE-SN,Rothenburg/Oberlausitz,0,NaN
23394,EDDB,large_airport,Berlin Brandenburg Airport,52.361738,13.502341,157.0,DE,DE-BR,Berlin,1,BER


## 5. Aggregate runway features per airport

From the runways table we compute three features for each airport:
- **runway_count**: how many runways the airport has
- **max_runway_length_ft**: length of the longest runway
- **primary_surface**: the most common surface material (e.g. ASP, CON)

In [5]:
# Only keep runways that belong to our filtered airports
our_icao_codes = set(airports["icao_code"])
runways = runways_raw[runways_raw["airport_ident"].isin(our_icao_codes)].copy()

# Aggregate
runway_agg = runways.groupby("airport_ident").agg(
    runway_count=("id", "count"),
    max_runway_length_ft=("length_ft", "max"),
).reset_index()

# Most common surface per airport
surface_mode = (
    runways.groupby("airport_ident")["surface"]
    .agg(lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else "Unknown")
    .reset_index()
    .rename(columns={"surface": "primary_surface"})
)

runway_agg = runway_agg.merge(surface_mode, on="airport_ident", how="left")
runway_agg = runway_agg.rename(columns={"airport_ident": "icao_code"})

print(f"Runway features computed for {len(runway_agg)} airports")
runway_agg.head()

Runway features computed for 365 airports


,icao_code,runway_count,max_runway_length_ft,primary_surface
0,EDAC,1,7988.0,CON
1,EDAH,3,7562.0,GRS
2,EDBN,1,7522.0,CON
3,EDBR,2,6562.0,ASP
4,EDDB,2,13123.0,ASPH


## 6. Download and load EUROCONTROL data

We download two datasets from the EUROCONTROL ANS Performance portal:

1. **Airport Arrival ATFM Delays** — daily ATFM delay minutes and flights
   delayed beyond 15 minutes, per airport. This is our primary delay source.

2. **Airport Traffic** — daily departure and arrival counts per airport.
   This gives us the departure volume that the delay file does not include.

Both files are available at stable URLs. The delay files are bz2-compressed;
pandas reads them directly without any extra steps.

In [6]:
# --- Load ATFM delay data ---
dly_frames = []
for url in ATFM_DLY_URLS:
    print(f"Loading {url.split('/')[-1]} ... ", end="")
    df = pd.read_csv(url)
    df = df[df["STATE_NAME"].isin(TARGET_COUNTRIES)]
    dly_frames.append(df)
    print(f"{len(df)} rows")

delays = pd.concat(dly_frames, ignore_index=True)
print(f"\nTotal ATFM delay rows (5 countries, 2023-2024): {len(delays)}")
print(f"Airports: {delays['APT_ICAO'].nunique()}")

Loading apt_dly_2023.csv.bz2 ... 

53575 rows
Loading apt_dly_2024.csv.bz2 ... 

53705 rows

Total ATFM delay rows (5 countries, 2023-2024): 107280
Airports: 155


In [7]:
# --- Load airport traffic data ---
trf_frames = []
for url in TRAFFIC_URLS:
    print(f"Loading {url.split('/')[-1]} ... ", end="")
    df = pd.read_csv(url)
    df = df[df["STATE_NAME"].isin(TARGET_COUNTRIES)]
    trf_frames.append(df)
    print(f"{len(df)} rows")

traffic = pd.concat(trf_frames, ignore_index=True)
print(f"\nTotal traffic rows (5 countries, 2023-2024): {len(traffic)}")

Loading airport_traffic_2023.csv ... 

54031 rows
Loading airport_traffic_2024.csv ... 

54169 rows

Total traffic rows (5 countries, 2023-2024): 108200


## 7. Standardise EUROCONTROL columns

We rename columns to snake_case and parse the date field.
We also compute two key derived metrics from the delay data:
- **avg_atfm_delay_min**: average ATFM arrival delay per flight
- **pct_delayed_15**: percentage of flights arriving more than 15 minutes late

In [8]:
# --- Standardise delay data ---
delays = delays.rename(columns={
    "APT_ICAO": "icao_code",
    "APT_NAME": "apt_name",
    "STATE_NAME": "country",
    "FLT_DATE": "date",
    "FLT_ARR_1": "arrivals",
    "DLY_APT_ARR_1": "total_atfm_delay_min",
    "FLT_ARR_1_DLY": "flights_with_delay",
    "FLT_ARR_1_DLY_15": "flights_delayed_15",
    "DLY_APT_ARR_W_1": "delay_weather_min",
    "DLY_APT_ARR_C_1": "delay_capacity_min",
    "DLY_APT_ARR_S_1": "delay_staffing_min",
})

# Parse date
delays["date"] = pd.to_datetime(delays["date"], utc=True).dt.date
delays["date"] = pd.to_datetime(delays["date"])

# Compute per-flight averages
delays["avg_atfm_delay_min"] = delays["total_atfm_delay_min"] / delays["arrivals"]
delays["pct_delayed_15"] = (delays["flights_delayed_15"] / delays["arrivals"]) * 100

# Keep only the columns we need going forward
delay_cols = [
    "icao_code", "apt_name", "country", "date",
    "arrivals", "total_atfm_delay_min", "flights_with_delay",
    "flights_delayed_15", "avg_atfm_delay_min", "pct_delayed_15",
    "delay_weather_min", "delay_capacity_min", "delay_staffing_min",
]
delays = delays[delay_cols].copy()

print(f"Delay data shape: {delays.shape}")
delays.head()

Delay data shape: (107280, 13)


,icao_code,apt_name,country,date,arrivals,total_atfm_delay_min,flights_with_delay,flights_delayed_15,avg_atfm_delay_min,pct_delayed_15,delay_weather_min,delay_capacity_min,delay_staffing_min
0,EGPH,Edinburgh,United Kingdom,2023-01-01,105,77,4,2,0.733333,1.904762,0,0,0
1,EDDP,Leipzig-Halle,Germany,2023-01-03,102,116,16,1,1.137255,0.980392,116,0,0
2,GCTS,Tenerife Sur/Reina Sofia,Spain,2023-01-03,145,686,43,19,4.731034,13.103448,0,686,0
3,LFBO,Toulouse-Blagnac,France,2023-01-04,96,163,14,5,1.697917,5.208333,163,0,0
4,LFPO,Paris-Orly,France,2023-01-07,248,572,36,11,2.306452,4.435484,0,0,0


In [9]:
# --- Standardise traffic data ---
traffic = traffic.rename(columns={
    "APT_ICAO": "icao_code",
    "FLT_DATE": "date",
    "FLT_DEP_1": "departures",
    "FLT_TOT_1": "total_movements",
})

traffic["date"] = pd.to_datetime(traffic["date"])

# Keep only what we need (departures and total movements)
traffic = traffic[["icao_code", "date", "departures", "total_movements"]].copy()

print(f"Traffic data shape: {traffic.shape}")
traffic.head()

Traffic data shape: (108200, 4)


,icao_code,date,departures,total_movements
0,LFBD,2023-01-01,63,126
1,LFBH,2023-01-01,1,2
2,LFBI,2023-01-01,1,2
3,LFBL,2023-01-01,2,5
4,LFBO,2023-01-01,81,161


## 8. Merge all data sources

We build the analytical dataset in three merge steps:
1. **Delays + Traffic** on (icao_code, date) — adds departure counts
2. **Result + Airport metadata** on icao_code — adds type, lat, lon, elevation, etc.
3. **Result + Runway features** on icao_code — adds runway count and length

In [10]:
# Step 1: Merge delays with traffic
merged = delays.merge(
    traffic,
    on=["icao_code", "date"],
    how="left"
)
print(f"After delays + traffic merge: {merged.shape}")

# Step 2: Merge with airport metadata
merged = merged.merge(
    airports,
    on="icao_code",
    how="left"
)
print(f"After airport metadata merge: {merged.shape}")

# Step 3: Merge with runway features
merged = merged.merge(
    runway_agg,
    on="icao_code",
    how="left"
)
print(f"After runway features merge: {merged.shape}")

# Check how many rows matched airport metadata
matched = merged["type"].notna().sum()
print(f"\nRows with airport metadata: {matched}/{len(merged)} ({matched/len(merged)*100:.1f}%)")

After delays + traffic merge: (107280, 15)
After airport metadata merge: (107280, 25)
After runway features merge: (107280, 28)

Rows with airport metadata: 103466/107280 (96.4%)


## 9. Feature engineering

We derive temporal and categorical features from the merged data. These are
the features that will let the model pick up seasonal patterns, weekday effects,
and airport-level differences.

In [11]:
# --- Temporal features ---
merged["year"] = merged["date"].dt.year
merged["month"] = merged["date"].dt.month
merged["quarter"] = merged["date"].dt.quarter
merged["weekday"] = merged["date"].dt.dayofweek  # 0=Mon, 6=Sun
merged["is_weekend"] = (merged["weekday"] >= 5).astype(int)

# Season mapping
season_map = {
    12: "winter", 1: "winter", 2: "winter",
    3: "spring", 4: "spring", 5: "spring",
    6: "summer", 7: "summer", 8: "summer",
    9: "autumn", 10: "autumn", 11: "autumn",
}
merged["season"] = merged["month"].map(season_map)
merged["is_summer"] = (merged["season"] == "summer").astype(int)

# --- Hub flag: airports in the top 25% by average daily arrivals ---
avg_arrivals = merged.groupby("icao_code")["arrivals"].mean()
hub_threshold = avg_arrivals.quantile(0.75)
hub_airports = set(avg_arrivals[avg_arrivals >= hub_threshold].index)
merged["hub_flag"] = merged["icao_code"].isin(hub_airports).astype(int)

# --- Traffic volume class ---
merged["traffic_volume_class"] = pd.qcut(
    merged["arrivals"], q=3, labels=["low", "medium", "high"]
)

print("Feature engineering complete.")
print(f"\nHub threshold: {hub_threshold:.0f} avg daily arrivals")
print(f"Hub airports: {len(hub_airports)} out of {merged['icao_code'].nunique()}")
print(f"\nNew columns added: year, month, quarter, weekday, is_weekend, season, is_summer, hub_flag, traffic_volume_class")

Feature engineering complete.



Hub threshold: 80 avg daily arrivals
Hub airports: 39 out of 155

New columns added: year, month, quarter, weekday, is_weekend, season, is_summer, hub_flag, traffic_volume_class


## 10. Validate the final dataset

We check the three hard requirements from the project brief:
- At least 1,000 rows
- At least 15 meaningful features
- At least 5 countries

In [12]:
print("=" * 50)
print("DATASET VALIDATION")
print("=" * 50)

n_rows = len(merged)
n_cols = merged.shape[1]
n_countries = merged["country"].nunique()

print(f"Rows       : {n_rows:,}  (need >= 1,000)  {'PASS' if n_rows >= 1000 else 'FAIL'}")
print(f"Columns    : {n_cols}  (need >= 15)     {'PASS' if n_cols >= 15 else 'FAIL'}")
print(f"Countries  : {n_countries}  (need >= 5)      {'PASS' if n_countries >= 5 else 'FAIL'}")
print()
print("Countries:")
print(merged["country"].value_counts())
print()
print("Date range:")
print(f"  From: {merged['date'].min()}")
print(f"  To:   {merged['date'].max()}")
print()
print("Airports per country:")
print(merged.groupby("country")["icao_code"].nunique())

# Hard assertions
assert n_rows >= 1000, f"Only {n_rows} rows!"
assert n_cols >= 15, f"Only {n_cols} columns!"
assert n_countries >= 5, f"Only {n_countries} countries!"

DATASET VALIDATION
Rows       : 107,280  (need >= 1,000)  PASS
Columns    : 37  (need >= 15)     PASS
Countries  : 5  (need >= 5)      PASS

Countries:
country
France            44181
Spain             34726
United Kingdom    13875
Germany           10941
Netherlands        3557
Name: count, dtype: int64

Date range:
  From: 2023-01-01 00:00:00
  To:   2024-12-31 00:00:00

Airports per country:
country
France            63
Germany           15
Netherlands        5
Spain             53
United Kingdom    19
Name: icao_code, dtype: int64


## 11. Save the merged raw dataset

In [13]:
out_path = PROCESSED_DIR / "merged_raw.csv"
merged.to_csv(out_path, index=False)

size_mb = out_path.stat().st_size / (1024 * 1024)
print(f"Saved: {out_path}")
print(f"Size:  {size_mb:.1f} MB")
print(f"Shape: {merged.shape}")

Saved: ..\data\processed\merged_raw.csv
Size:  21.2 MB
Shape: (107280, 37)


## 12. Column reference

Summary of all columns in the merged dataset and where they come from.

In [14]:
print(f"{'Column':<30} {'Dtype':<15} {'Non-Null':<12} {'Sample Value'}")
print("-" * 80)
for col in merged.columns:
    dtype = str(merged[col].dtype)
    non_null = merged[col].notna().sum()
    sample = merged[col].dropna().iloc[0] if merged[col].notna().any() else "NaN"
    print(f"{col:<30} {dtype:<15} {non_null:<12} {sample}")

Column                         Dtype           Non-Null     Sample Value
--------------------------------------------------------------------------------
icao_code                      str             107280       EGPH
apt_name                       str             107280       Edinburgh
country                        str             107280       United Kingdom
date                           datetime64[s]   107280       2023-01-01 00:00:00
arrivals                       int64           107280       105
total_atfm_delay_min           int64           107280       77
flights_with_delay             int64           107280       4
flights_delayed_15             int64           107280       2
avg_atfm_delay_min             float64         105193       0.7333333333333333
pct_delayed_15                 float64         105193       1.9047619047619049
delay_weather_min              int64           107280       0
delay_capacity_min             int64           107280       0
delay_staffing_min     

---

**Next step:** `02_cleaning.ipynb` will load `merged_raw.csv`, handle missing values,
fix data types, deal with outliers, and produce the final clean dataset for EDA.